In [5]:
import json
from pathlib import Path

import numpy as np
from sahi.slicing import slice_coco
from skmultilearn.model_selection import iterative_train_test_split

In [6]:
MSGO_CLASSES = {
    "Plane": 0,
    "Bridge": 1,
    "Intersection": 2,
    "Roundabout": 3,
    "Vehicle": 4,
    "Ship": 5,
}

NUM_CLASSES = len(MSGO_CLASSES)

In [7]:
def build_image_to_counts(root_dir: str) -> dict[str, dict[int, int]]:
    root_path = Path(root_dir)
    image_to_counts = {}

    images_dir = root_path / "images"
    labels_dir = root_path / "labels"

    for label_file in labels_dir.glob("*.txt"):
        img_file = images_dir / f"{label_file.stem}.jpg"

        counts = {}
        with open(label_file) as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            counts[class_id] = counts.get(class_id, 0) + 1

        image_to_counts[str(img_file)] = counts

    return image_to_counts

In [8]:
image_to_counts = build_image_to_counts("D:\\stuff\\datasets\\MSGOv1")

In [9]:
image_paths = list(image_to_counts.keys())
num_images = len(image_paths)

y_counts = np.zeros((num_images, NUM_CLASSES), dtype=int)
for i, path in enumerate(image_paths):
    counts = image_to_counts[path]
    for class_id, count in counts.items():
        y_counts[i, class_id] = count

X = np.array(image_paths).reshape(-1, 1)

In [ ]:
X_train, y_train_counts, X_temp, y_temp_counts = iterative_train_test_split(X, y_counts, test_size=0.2)
X_val, y_val_counts, X_test, y_test_counts = iterative_train_test_split(X_temp, y_temp_counts, test_size=0.5)

X_train_paths = X_train.flatten().tolist()
X_val_paths = X_val.flatten().tolist()
X_test_paths = X_test.flatten().tolist()

print(f"Total images: {len(X)}")
print(f"Train images: {len(X_train_paths)} ({len(X_train_paths) / len(X):.1%})")
print(f"Validation images: {len(X_val_paths)} ({len(X_val_paths) / len(X):.1%})")
print(f"Test images: {len(X_test_paths)} ({len(X_test_paths) / len(X):.1%})")

Total images: 39175
Train images: 31340 (80.0%)
Validation images: 3918 (10.0%)
Test images: 3917 (10.0%)


In [11]:
def check_distribution(paths, image_to_counts_map, num_classes):
    total_counts = np.zeros(num_classes, dtype=int)
    for path in paths:
        counts = image_to_counts_map.get(path, {})
        for class_id, count in counts.items():
            total_counts[class_id] += count
    return total_counts


train_counts = check_distribution(X_train_paths, image_to_counts, NUM_CLASSES)
val_counts = check_distribution(X_val_paths, image_to_counts, NUM_CLASSES)
test_counts = check_distribution(X_test_paths, image_to_counts, NUM_CLASSES)
total_counts = train_counts + val_counts + test_counts

print(f"Class Names: {list(MSGO_CLASSES.keys())}")
print(f"Total Instances: {total_counts}")
print(f"Train Instances: {train_counts} ({(train_counts / total_counts * 100).round(1)}%)")
print(f"Val Instances:   {val_counts} ({(val_counts / total_counts * 100).round(1)}%)")
print(f"Test Instances:  {test_counts} ({(test_counts / total_counts * 100).round(1)}%)")

Class Names: ['Plane', 'Bridge', 'Intersection', 'Roundabout', 'Vehicle', 'Ship']
Total Instances: [ 63950   8620  10156   1718 750564 174630]
Train Instances: [ 51906   6711   8145   1280 676816 129805] ([81.2 77.9 80.2 74.5 90.2 74.3]%)
Val Instances:   [ 5872  1005   992   251 37172 21534] ([ 9.2 11.7  9.8 14.6  5.  12.3]%)
Test Instances:  [ 6172   904  1019   187 36576 23291] ([ 9.7 10.5 10.  10.9  4.9 13.3]%)


In [ ]:
ROOT_DIR = Path("D:/stuff/datasets/MSGOv1")
MASTER_COCO_PATH = ROOT_DIR / "master_annotations.json"
FINAL_DATASET_DIR = ROOT_DIR / "sliced"
IMAGE_ROOT_DIR = ROOT_DIR


def slice_split(image_paths, master_coco_data, split_name):
    output_dir = FINAL_DATASET_DIR / split_name
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing {split_name} split...")

    split_relative_paths = {Path(p).relative_to(IMAGE_ROOT_DIR).as_posix() for p in image_paths}
    split_images = [img for img in master_coco_data["images"] if img["file_name"] in split_relative_paths]
    split_image_ids = {img["id"] for img in split_images}
    split_annotations = [ann for ann in master_coco_data["annotations"] if ann["image_id"] in split_image_ids]

    subset_coco_data = {
        "images": split_images,
        "annotations": split_annotations,
        "categories": master_coco_data["categories"],
    }

    subset_coco_path = output_dir / f"{split_name}_subset.json"
    with open(subset_coco_path, "w") as f:
        json.dump(subset_coco_data, f)

    slice_coco(
        coco_annotation_file_path=subset_coco_path,
        image_dir=IMAGE_ROOT_DIR,
        output_dir=output_dir,
        output_coco_annotation_file_name="sliced_annotations.json",
        slice_height=800,
        slice_width=800,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        min_area_ratio=0.1,
        ignore_negative_samples=False,
        verbose=False,
    )

    subset_coco_path.unlink()

In [ ]:
with open(MASTER_COCO_PATH) as f:
    master_data = json.load(f)

slice_split(X_test_paths, master_data, "test")
slice_split(X_val_paths, master_data, "val")
slice_split(X_train_paths, master_data, "train")

In [ ]:
import json
from pathlib import Path

MASTER_COCO_PATH = Path("D:/stuff/datasets/MSGOv1/master_annotations.json")

with open(MASTER_COCO_PATH) as f:
    master_data = json.load(f)

bad_annotations_found = 0
for annotation in master_data["annotations"]:
    bbox = annotation.get("bbox")
    area = annotation.get("area")

    if not isinstance(bbox, list) or len(bbox) != 4 or area <= 1:
        print(f"Found bad annotation (ID: {annotation['id']}):")
        print(json.dumps(annotation, indent=2))
        print(f"AREA: {area}")
        bad_annotations_found += 1


print(f"\nFound {bad_annotations_found} bad annotation(s).")